# End-to-End VQA Pipeline with Google Gemini

A unified, production-ready Visual Question Answering pipeline following a complete ML workflow:

```
Problem Definition → Data Input → Preprocessing → Prompt Engineering → 
Model Invocation → Response Generation → Post-processing → Evaluation → 
Storage/Logging → Deployment
```

## Table of Contents

1. [Problem Definition](#1-Problem-Definition)
2. [Data Input Layer](#2-Data-Input-Layer)
3. [Preprocessing](#3-Preprocessing)
4. [Prompt Engineering](#4-Prompt-Engineering)
5. [Model Invocation (Gemini API)](#5-Model-Invocation-Gemini-API)
6. [Response Generation](#6-Response-Generation)
7. [Post-processing](#7-Post-processing)
8. [Evaluation](#8-Evaluation)
9. [Storage / Logging](#9-Storage--Logging)
10. [Deployment (API/UI)](#10-Deployment-APIUI)

## Setup

In [ ]:
# Core dependencies
import os
import io
import base64
import json
import logging
import hashlib
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple
from dataclasses import dataclass, asdict
from enum import Enum

from PIL import Image
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Load environment variables
load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logger = logging.getLogger("vqa_pipeline")

# Verify API key
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found. Set it in your .env file")

logger.info("✓ Environment loaded successfully")

---

## 1. Problem Definition

Define the scope, constraints, and objectives of the VQA system.

In [ ]:
class TaskType(Enum):
    """Supported VQA task types."""
    GENERAL = "general"           # Open-ended questions
    CLASSIFICATION = "classification"  # Multiple choice / categories
    COUNTING = "counting"         # Count objects
    LOCALIZATION = "localization" # Where is X?
    OCR = "ocr"                   # Read text in image
    REASONING = "reasoning"       # Complex multi-step reasoning


@dataclass
class ProblemConfig:
    """Defines the VQA problem scope and constraints."""
    
    # Task configuration
    task_type: TaskType = TaskType.GENERAL
    domain: str = "general"       # e.g., "medical", "retail", "general"
    
    # Quality constraints
    max_response_length: int = 500  # tokens
    confidence_threshold: float = 0.7
    require_citation: bool = False  # Cite visual evidence
    
    # Performance constraints
    max_latency_ms: int = 5000
    max_image_pixels: int = 20_000_000
    max_image_size_mb: int = 10
    
    # Model configuration
    model_name: str = "gemini-2.5-flash"
    temperature: float = 0.3
    
    # Output configuration
    output_format: str = "text"   # text, json, structured
    include_reasoning: bool = False
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            **asdict(self),
            "task_type": self.task_type.value
        }


# Define problem for this session
problem_config = ProblemConfig(
    task_type=TaskType.GENERAL,
    domain="general",
    max_response_length=500,
    temperature=0.3,
    include_reasoning=False
)

logger.info(f"✓ Problem defined: {problem_config.task_type.value} VQA in {problem_config.domain} domain")
logger.info(f"  Model: {problem_config.model_name}, Temp: {problem_config.temperature}")

---

## 2. Data Input Layer

Handles loading images from various sources (file, URL, base64, bytes).

In [ ]:
@dataclass
class ImageInput:
    """Normalized image input with metadata."""
    pil_image: Image.Image
    source: str                    # file path, URL, or "base64"
    original_format: str           # JPEG, PNG, etc.
    original_size: Tuple[int, int] # (width, height)
    source_bytes: Optional[int] = None
    checksum: Optional[str] = None


class DataInputLayer:
    """Unified image input handler."""
    
    ALLOWED_FORMATS = {"JPEG", "PNG", "WEBP", "BMP", "GIF"}
    
    def load_from_file(self, path: str) -> ImageInput:
        """Load image from file path."""
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Image not found: {path}")
        
        source_bytes = path.stat().st_size
        image = Image.open(path)
        image.verify()
        image = Image.open(path)  # Re-open after verify
        
        checksum = hashlib.md5(path.read_bytes()).hexdigest()[:12]
        
        return ImageInput(
            pil_image=image,
            source=str(path),
            original_format=image.format or "UNKNOWN",
            original_size=image.size,
            source_bytes=source_bytes,
            checksum=checksum
        )
    
    def load_from_bytes(self, data: bytes, source: str = "bytes") -> ImageInput:
        """Load image from bytes."""
        image = Image.open(io.BytesIO(data))
        image.verify()
        image = Image.open(io.BytesIO(data))
        
        checksum = hashlib.md5(data).hexdigest()[:12]
        
        return ImageInput(
            pil_image=image,
            source=source,
            original_format=image.format or "UNKNOWN",
            original_size=image.size,
            source_bytes=len(data),
            checksum=checksum
        )
    
    def load_from_base64(self, b64_string: str) -> ImageInput:
        """Load image from base64 string."""
        # Handle data URI format
        if "," in b64_string:
            b64_string = b64_string.split(",")[1]
        
        data = base64.b64decode(b64_string)
        return self.load_from_bytes(data, source="base64")


input_layer = DataInputLayer()
logger.info("✓ Data Input Layer initialized")

---

## 3. Preprocessing

Image validation, normalization, and enhancement.

In [ ]:
@dataclass
class PreprocessingResult:
    """Result of preprocessing stage."""
    image: Image.Image
    transformations: List[str]     # Applied transformations
    validation_passed: bool
    validation_errors: List[str]
    final_size: Tuple[int, int]
    final_format: str


class PreprocessingStage:
    """Image preprocessing with validation and enhancement."""
    
    def __init__(self, config: ProblemConfig):
        self.config = config
        self.allowed_formats = config.allowed_formats if hasattr(config, 'allowed_formats') else {"JPEG", "PNG", "WEBP"}
    
    def validate(self, image_input: ImageInput) -> Tuple[bool, List[str]]:
        """Validate image against constraints."""
        errors = []
        
        # Check format
        if image_input.original_format not in self.allowed_formats:
            errors.append(f"Unsupported format: {image_input.original_format}")
        
        # Check dimensions
        w, h = image_input.original_size
        if w * h > self.config.max_image_pixels:
            errors.append(f"Image too large: {w}x{h} > {self.config.max_image_pixels} pixels")
        
        # Check file size
        if image_input.source_bytes:
            size_mb = image_input.source_bytes / (1024 * 1024)
            if size_mb > self.config.max_image_size_mb:
                errors.append(f"File too large: {size_mb:.1f}MB > {self.config.max_image_size_mb}MB")
        
        return len(errors) == 0, errors
    
    def transform(self, image: Image.Image) -> Tuple[Image.Image, List[str]]:
        """Apply transformations."""
        transforms = []
        
        # Convert to RGB (handles RGBA, P, LA modes)
        if image.mode != "RGB":
            image = image.convert("RGB")
            transforms.append("converted_to_RGB")
        
        # Auto-orient (handle EXIF rotation)
        try:
            from PIL import ImageOps
            image = ImageOps.exif_transpose(image)
            transforms.append("exif_oriented")
        except Exception:
            pass  # No EXIF or unsupported
        
        # Resize if needed
        w, h = image.size
        max_dim = int(self.config.max_image_pixels ** 0.5)
        if max(w, h) > max_dim:
            scale = max_dim / max(w, h)
            new_size = (int(w * scale), int(h * scale))
            image = image.resize(new_size, Image.Resampling.LANCZOS)
            transforms.append(f"resized_to_{new_size[0]}x{new_size[1]}")
        
        return image, transforms
    
    def process(self, image_input: ImageInput) -> PreprocessingResult:
        """Full preprocessing pipeline."""
        # Validate
        valid, errors = self.validate(image_input)
        
        if not valid:
            return PreprocessingResult(
                image=image_input.pil_image,
                transformations=[],
                validation_passed=False,
                validation_errors=errors,
                final_size=image_input.original_size,
                final_format=image_input.original_format
            )
        
        # Transform
        image, transforms = self.transform(image_input.pil_image)
        
        return PreprocessingResult(
            image=image,
            transformations=transforms,
            validation_passed=True,
            validation_errors=[],
            final_size=image.size,
            final_format="JPEG"
        )


preprocessing = PreprocessingStage(problem_config)
logger.info("✓ Preprocessing stage initialized")

---

## 4. Prompt Engineering

Construct optimized prompts based on task type and domain.

In [ ]:
@dataclass
class PromptTemplate:
    """Structured prompt with system and user components."""
    system_prompt: str
    user_prompt: str
    constraints: List[str]


class PromptEngineeringStage:
    """Dynamic prompt generation based on task and domain."""
    
    BASE_SYSTEM_PROMPTS = {
        TaskType.GENERAL: """You are a helpful visual assistant. Analyze the image and answer the question accurately.""",
        TaskType.CLASSIFICATION: """You are an image classifier. Identify the category that best matches the image content.""",
        TaskType.COUNTING: """You are a counting assistant. Count the specified objects carefully and provide only the number.""",
        TaskType.LOCALIZATION: """You are a spatial reasoning assistant. Describe the location of objects in the image.""",
        TaskType.OCR: """You are an OCR assistant. Read and transcribe all visible text in the image accurately.""",
        TaskType.REASONING: """You are a visual reasoning expert. Analyze the image step-by-step before answering.""",
    }
    
    DOMAIN_ENHANCEMENTS = {
        "medical": "Be precise with medical terminology. Note uncertainty when present.",
        "retail": "Focus on product details, brand names, and visual attributes.",
        "document": "Pay attention to text layout, formatting, and document structure.",
        "general": "",
    }
    
    def __init__(self, config: ProblemConfig):
        self.config = config
    
    def build_system_prompt(self) -> str:
        """Construct system prompt based on config."""
        base = self.BASE_SYSTEM_PROMPTS.get(self.config.task_type, self.BASE_SYSTEM_PROMPTS[TaskType.GENERAL])
        domain = self.DOMAIN_ENHANCEMENTS.get(self.config.domain, "")
        
        parts = [base]
        if domain:
            parts.append(domain)
        
        # Add constraints
        constraints = []
        if self.config.max_response_length:
            constraints.append(f"Keep responses under {self.config.max_response_length} tokens.")
        if self.config.require_citation:
            constraints.append("Cite specific visual evidence from the image.")
        if self.config.include_reasoning:
            constraints.append("Show your reasoning step-by-step.")
        
        if constraints:
            parts.append("\n\nConstraints:\n" + "\n".join(constraints))
        
        return "\n\n".join(parts)
    
    def build_user_prompt(self, question: str) -> str:
        """Build user prompt from question."""
        if self.config.task_type == TaskType.COUNTING:
            return f"Count and provide only the number: {question}"
        elif self.config.task_type == TaskType.CLASSIFICATION:
            return f"Classify: {question}"
        else:
            return question
    
    def create(self, question: str) -> PromptTemplate:
        """Create complete prompt template."""
        return PromptTemplate(
            system_prompt=self.build_system_prompt(),
            user_prompt=self.build_user_prompt(question),
            constraints=[]
        )


prompt_engineering = PromptEngineeringStage(problem_config)
logger.info("✓ Prompt Engineering stage initialized")

---

## 5. Model Invocation (Gemini API)

Configure and invoke the Gemini model with proper error handling.

In [ ]:
@dataclass
class ModelInvocationResult:
    """Result from model invocation."""
    raw_response: Any
    content: str
    latency_ms: float
    token_usage: Optional[Dict[str, int]] = None
    error: Optional[str] = None


class ModelInvocationStage:
    """Gemini API invocation with retry and error handling."""
    
    def __init__(self, config: ProblemConfig):
        self.config = config
        self.llm = ChatGoogleGenerativeAI(
            model=config.model_name,
            temperature=config.temperature
        )
    
    def encode_for_api(self, image: Image.Image) -> str:
        """Encode image for Gemini API."""
        buffered = io.BytesIO()
        image.save(buffered, format="JPEG", quality=85)
        return base64.b64encode(buffered.getvalue()).decode("utf-8")
    
    def invoke(self, image: Image.Image, prompt: PromptTemplate) -> ModelInvocationResult:
        """Invoke Gemini API with timing and error handling."""
        import time
        
        start_time = time.perf_counter()
        
        try:
            # Encode image
            image_b64 = self.encode_for_api(image)
            
            # Build message
            messages = [SystemMessage(content=prompt.system_prompt)]
            user_content = [
                {"type": "text", "text": prompt.user_prompt},
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{image_b64}"
                }
            ]
            messages.append(HumanMessage(content=user_content))
            
            # Invoke
            response = self.llm.invoke(messages)
            
            latency_ms = (time.perf_counter() - start_time) * 1000
            
            return ModelInvocationResult(
                raw_response=response,
                content=response.content,
                latency_ms=latency_ms,
                token_usage=None  # Gemini doesn't expose token count directly
            )
            
        except Exception as e:
            latency_ms = (time.perf_counter() - start_time) * 1000
            logger.error(f"Model invocation error: {e}")
            return ModelInvocationResult(
                raw_response=None,
                content="",
                latency_ms=latency_ms,
                error=str(e)
            )


model_invocation = ModelInvocationStage(problem_config)
logger.info(f"✓ Model Invocation stage initialized ({problem_config.model_name})")

---

## 6. Response Generation

Parse and structure the model output.

In [ ]:
@dataclass
class GeneratedResponse:
    """Structured response output."""
    answer: str
    confidence: Optional[float] = None
    reasoning: Optional[str] = None
    citations: Optional[List[str]] = None
    metadata: Dict[str, Any] = None
    
    def __post_init__(self):
        if self.metadata is None:
            self.metadata = {}


class ResponseGenerationStage:
    """Parse and structure model output."""
    
    def __init__(self, config: ProblemConfig):
        self.config = config
    
    def parse_response(self, invocation_result: ModelInvocationResult,
                       prompt: PromptTemplate) -> GeneratedResponse:
        """Parse raw model output into structured response."""
        
        if invocation_result.error:
            return GeneratedResponse(
                answer=f"Error: {invocation_result.error}",
                metadata={"error": invocation_result.error}
            )
        
        content = invocation_result.content.strip()
        
        # Extract reasoning if requested
        reasoning = None
        if self.config.include_reasoning:
            # Simple parsing - can be enhanced with regex or secondary LLM call
            if "**Reasoning**:" in content:
                parts = content.split("**Reasoning**:")
                reasoning = parts[1].split("**")[0].strip() if len(parts) > 1 else None
        
        # Extract confidence if present
        confidence = None
        # Look for patterns like "Confidence: 0.85" or "(85% confident)"
        import re
        conf_match = re.search(r"[Cc]onfidence[:\s]+([0-9.]+)", content)
        if conf_match:
            confidence = float(conf_match.group(1))
        
        return GeneratedResponse(
            answer=content,
            confidence=confidence,
            reasoning=reasoning,
            metadata={
                "latency_ms": invocation_result.latency_ms,
                "prompt_type": self.config.task_type.value
            }
        )


response_generation = ResponseGenerationStage(problem_config)
logger.info("✓ Response Generation stage initialized")

---

## 7. Post-processing

Clean, format, and validate the response.

In [ ]:
@dataclass
class PostProcessingResult:
    """Final processed response."""
    answer: str
    formatted: bool
    truncated: bool
    validation_passed: bool
    warnings: List[str]


class PostProcessingStage:
    """Response cleanup and formatting."""
    
    def __init__(self, config: ProblemConfig):
        self.config = config
    
    def truncate(self, text: str, max_tokens: int) -> Tuple[str, bool]:
        """Truncate text to max tokens."""
        # Rough token estimate: 4 chars per token
        max_chars = max_tokens * 4
        if len(text) <= max_chars:
            return text, False
        
        truncated = text[:max_chars - 3] + "..."
        return truncated, True
    
    def clean(self, text: str) -> str:
        """Clean response text."""
        # Remove extra whitespace
        text = " ".join(text.split())
        
        # Remove common filler phrases
        fillers = [
            "Based on the image, ",
            "Looking at the image, ",
            "In the image, ",
            "The image shows ",
        ]
        for filler in fillers:
            if text.startswith(filler):
                text = text[len(filler):]
        
        return text.strip()
    
    def validate(self, response: GeneratedResponse) -> Tuple[bool, List[str]]:
        """Validate response against constraints."""
        warnings = []
        
        # Check length
        estimated_tokens = len(response.answer) // 4
        if estimated_tokens > self.config.max_response_length:
            warnings.append(f"Response exceeds max length ({estimated_tokens} > {self.config.max_response_length} tokens)")
        
        # Check confidence threshold
        if response.confidence and response.confidence < self.config.confidence_threshold:
            warnings.append(f"Low confidence: {response.confidence:.2f} < {self.config.confidence_threshold}")
        
        # Check for empty response
        if not response.answer.strip():
            return False, ["Empty response"]
        
        return True, warnings
    
    def process(self, response: GeneratedResponse) -> PostProcessingResult:
        """Full post-processing pipeline."""
        warnings = []
        
        # Validate
        valid, validation_warnings = self.validate(response)
        warnings.extend(validation_warnings)
        
        # Clean
        answer = self.clean(response.answer)
        
        # Truncate if needed
        answer, truncated = self.truncate(answer, self.config.max_response_length)
        if truncated:
            warnings.append("Response was truncated to max length")
        
        return PostProcessingResult(
            answer=answer,
            formatted=True,
            truncated=truncated,
            validation_passed=valid,
            warnings=warnings
        )


post_processing = PostProcessingStage(problem_config)
logger.info("✓ Post-processing stage initialized")

---

## 8. Evaluation

Assess response quality and performance metrics.

In [ ]:
@dataclass
class EvaluationMetrics:
    """Evaluation metrics for a single prediction."""
    latency_ms: float
    response_length: int
    estimated_tokens: int
    quality_score: Optional[float] = None
    matches_expected: Optional[bool] = None
    warnings: List[str] = None
    
    def __post_init__(self):
        if self.warnings is None:
            self.warnings = []


class EvaluationStage:
    """Response quality and performance evaluation."""
    
    def __init__(self, config: ProblemConfig):
        self.config = config
    
    def evaluate(self, 
                 post_result: PostProcessingResult,
                 invocation_result: ModelInvocationResult,
                 expected_answer: Optional[str] = None) -> EvaluationMetrics:
        """Compute evaluation metrics."""
        
        # Basic metrics
        response_length = len(post_result.answer)
        estimated_tokens = response_length // 4
        
        # Check against expected (if provided)
        matches_expected = None
        if expected_answer:
            # Simple string matching (can be enhanced with semantic similarity)
            matches_expected = expected_answer.lower() in post_result.answer.lower()
        
        # Quality score (heuristic)
        quality_score = None
        if post_result.validation_passed and not post_result.truncated:
            quality_score = 1.0
            if post_result.warnings:
                quality_score -= 0.1 * len(post_result.warnings)
            quality_score = max(0.0, quality_score)
        
        return EvaluationMetrics(
            latency_ms=invocation_result.latency_ms,
            response_length=response_length,
            estimated_tokens=estimated_tokens,
            quality_score=quality_score,
            matches_expected=matches_expected,
            warnings=post_result.warnings
        )
    
    def aggregate(self, metrics: List[EvaluationMetrics]) -> Dict[str, Any]:
        """Aggregate metrics across multiple predictions."""
        if not metrics:
            return {}
        
        latencies = [m.latency_ms for m in metrics]
        tokens = [m.estimated_tokens for m in metrics]
        quality_scores = [m.quality_score for m in metrics if m.quality_score is not None]
        matches = [m.matches_expected for m in metrics if m.matches_expected is not None]
        
        return {
            "count": len(metrics),
            "avg_latency_ms": sum(latencies) / len(latencies),
            "max_latency_ms": max(latencies),
            "min_latency_ms": min(latencies),
            "avg_tokens": sum(tokens) / len(tokens),
            "avg_quality_score": sum(quality_scores) / len(quality_scores) if quality_scores else None,
            "accuracy": sum(matches) / len(matches) if matches else None,
            "success_rate": sum(1 for m in metrics if m.quality_score and m.quality_score > 0.5) / len(metrics)
        }


evaluation = EvaluationStage(problem_config)
logger.info("✓ Evaluation stage initialized")

---

## 9. Storage / Logging

Persist predictions, metrics, and audit trails.

In [ ]:
@dataclass
class PredictionRecord:
    """Complete prediction record for storage."""
    id: str
    timestamp: str
    image_source: str
    image_checksum: Optional[str]
    question: str
    answer: str
    expected_answer: Optional[str]
    metrics: Dict[str, Any]
    config: Dict[str, Any]
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


class StorageLoggingStage:
    """Prediction storage and logging."""
    
    def __init__(self, output_dir: str = "./pipeline_logs"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.records: List[PredictionRecord] = []
    
    def generate_id(self, image_checksum: str, question: str) -> str:
        """Generate unique prediction ID."""
        content = f"{image_checksum}:{question}:{datetime.now().isoformat()}"
        return hashlib.sha256(content.encode()).hexdigest()[:16]
    
    def record(self,
               image_input: ImageInput,
               question: str,
               answer: str,
               expected_answer: Optional[str],
               metrics: EvaluationMetrics,
               config: ProblemConfig) -> PredictionRecord:
        """Create and store prediction record."""
        
        record = PredictionRecord(
            id=self.generate_id(image_input.checksum or "", question),
            timestamp=datetime.now().isoformat(),
            image_source=image_input.source,
            image_checksum=image_input.checksum,
            question=question,
            answer=answer,
            expected_answer=expected_answer,
            metrics={
                "latency_ms": metrics.latency_ms,
                "response_length": metrics.response_length,
                "estimated_tokens": metrics.estimated_tokens,
                "quality_score": metrics.quality_score,
                "matches_expected": metrics.matches_expected,
                "warnings": metrics.warnings
            },
            config=config.to_dict()
        )
        
        self.records.append(record)
        logger.info(f"✓ Recorded prediction {record.id}")
        return record
    
    def save_to_json(self, filepath: Optional[str] = None) -> str:
        """Save all records to JSON file."""
        if filepath is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filepath = str(self.output_dir / f"predictions_{timestamp}.json")
        
        data = [r.to_dict() for r in self.records]
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        logger.info(f"✓ Saved {len(self.records)} records to {filepath}")
        return filepath
    
    def export_csv(self, filepath: Optional[str] = None) -> str:
        """Export records to CSV."""
        import csv
        
        if filepath is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filepath = str(self.output_dir / f"predictions_{timestamp}.csv")
        
        if not self.records:
            raise ValueError("No records to export")
        
        with open(filepath, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "id", "timestamp", "image_source", "question", "answer",
                "expected_answer", "latency_ms", "quality_score", "matches_expected"
            ])
            writer.writeheader()
            for r in self.records:
                writer.writerow({
                    "id": r.id,
                    "timestamp": r.timestamp,
                    "image_source": r.image_source,
                    "question": r.question,
                    "answer": r.answer,
                    "expected_answer": r.expected_answer,
                    "latency_ms": r.metrics["latency_ms"],
                    "quality_score": r.metrics["quality_score"],
                    "matches_expected": r.metrics["matches_expected"]
                })
        
        logger.info(f"✓ Exported {len(self.records)} records to CSV: {filepath}")
        return filepath


storage = StorageLoggingStage()
logger.info("✓ Storage/Logging stage initialized")

---

## 10. Deployment (API/UI)

Integrate the pipeline into Flask API and web UI.

In [ ]:
class UnifiedVQAPipeline:
    """
    Complete VQA pipeline integrating all stages.
    
    Usage:
        pipeline = UnifiedVQAPipeline()
        result = pipeline.predict(image_path, "What is in this image?")
    """
    
    def __init__(self, config: Optional[ProblemConfig] = None):
        self.config = config or ProblemConfig()
        
        # Initialize all stages
        self.input_layer = DataInputLayer()
        self.preprocessing = PreprocessingStage(self.config)
        self.prompt_engineering = PromptEngineeringStage(self.config)
        self.model_invocation = ModelInvocationStage(self.config)
        self.response_generation = ResponseGenerationStage(self.config)
        self.post_processing = PostProcessingStage(self.config)
        self.evaluation = EvaluationStage(self.config)
        self.storage = StorageLoggingStage()
        
        logger.info(f"✓ UnifiedVQAPipeline initialized ({self.config.model_name})")
    
    def predict(self, 
                image_source: str,
                question: str,
                expected_answer: Optional[str] = None,
                save_record: bool = True) -> Dict[str, Any]:
        """
        Run complete VQA pipeline.
        
        Args:
            image_source: File path, URL, or base64 string
            question: The question to answer
            expected_answer: Optional ground truth for evaluation
            save_record: Whether to store the prediction record
        
        Returns:
            Dictionary with answer, metrics, and metadata
        """
        logger.info(f"Processing: {question}")
        
        # Stage 1: Data Input
        if Path(image_source).exists():
            image_input = self.input_layer.load_from_file(image_source)
        else:
            image_input = self.input_layer.load_from_base64(image_source)
        logger.debug(f"  Input: {image_input.original_format} {image_input.original_size}")
        
        # Stage 2: Preprocessing
        prep_result = self.preprocessing.process(image_input)
        if not prep_result.validation_passed:
            logger.error(f"  Preprocessing failed: {prep_result.validation_errors}")
            return {
                "answer": "Image validation failed",
                "errors": prep_result.validation_errors,
                "success": False
            }
        logger.debug(f"  Preprocessed: {prep_result.transformations}")
        
        # Stage 3: Prompt Engineering
        prompt = self.prompt_engineering.create(question)
        logger.debug(f"  Prompt type: {self.config.task_type.value}")
        
        # Stage 4: Model Invocation
        invocation_result = self.model_invocation.invoke(prep_result.image, prompt)
        if invocation_result.error:
            logger.error(f"  Model error: {invocation_result.error}")
            return {
                "answer": f"Model error: {invocation_result.error}",
                "success": False
            }
        logger.debug(f"  Latency: {invocation_result.latency_ms:.0f}ms")
        
        # Stage 5: Response Generation
        response = self.response_generation.parse_response(invocation_result, prompt)
        
        # Stage 6: Post-processing
        post_result = self.post_processing.process(response)
        logger.debug(f"  Post-processed: truncated={post_result.truncated}")
        
        # Stage 7: Evaluation
        metrics = self.evaluation.evaluate(post_result, invocation_result, expected_answer)
        
        # Stage 8: Storage
        record = None
        if save_record:
            record = self.storage.record(
                image_input=image_input,
                question=question,
                answer=post_result.answer,
                expected_answer=expected_answer,
                metrics=metrics,
                config=self.config
            )
        
        # Return result
        result = {
            "answer": post_result.answer,
            "success": True,
            "metrics": {
                "latency_ms": metrics.latency_ms,
                "response_length": metrics.response_length,
                "quality_score": metrics.quality_score,
                "matches_expected": metrics.matches_expected
            },
            "warnings": post_result.warnings,
            "record_id": record.id if record else None
        }
        
        logger.info(f"✓ Complete: {result['metrics']['latency_ms']:.0f}ms")
        return result
    
    def predict_batch(self,
                      samples: List[Tuple[str, str, Optional[str]]],
                      save_records: bool = True) -> List[Dict[str, Any]]:
        """
        Run pipeline on multiple samples.
        
        Args:
            samples: List of (image_path, question, expected_answer) tuples
            save_records: Whether to store prediction records
        
        Returns:
            List of result dictionaries
        """
        results = []
        for image_path, question, expected in samples:
            result = self.predict(image_path, question, expected, save_records)
            results.append(result)
        return results
    
    def get_metrics_summary(self) -> Dict[str, Any]:
        """Get aggregated metrics from all predictions."""
        records = self.storage.records
        if not records:
            return {}
        
        metrics_list = [
            EvaluationMetrics(
                latency_ms=r.metrics["latency_ms"],
                response_length=r.metrics["response_length"],
                estimated_tokens=r.metrics["estimated_tokens"],
                quality_score=r.metrics["quality_score"],
                matches_expected=r.metrics["matches_expected"],
                warnings=r.metrics["warnings"]
            )
            for r in records
        ]
        
        return self.evaluation.aggregate(metrics_list)
    
    def save_results(self, filepath: Optional[str] = None) -> str:
        """Save all prediction records to JSON."""
        return self.storage.save_to_json(filepath)


# Initialize the unified pipeline
pipeline = UnifiedVQAPipeline()
logger.info("✓ Unified VQA Pipeline ready for deployment")

### Flask API Integration

In [ ]:
# Flask API wrapper for the pipeline
# This mirrors the production app.py structure

from flask import Flask, request, jsonify

def create_flask_app(pipeline: UnifiedVQAPipeline) -> Flask:
    """Create Flask app with VQA pipeline."""
    app = Flask(__name__)
    app.config["MAX_CONTENT_LENGTH"] = 10 * 1024 * 1024  # 10MB
    
    @app.route("/")
    def index():
        return jsonify({"status": "ready", "model": pipeline.config.model_name})
    
    @app.route("/predict", methods=["POST"])
    def predict():
        if "image" not in request.files:
            return jsonify({"error": "No image provided"}), 400
        
        file = request.files["image"]
        question = request.form.get("question", "")
        
        if not question:
            return jsonify({"error": "No question provided"}), 400
        
        # Save uploaded file temporarily
        import tempfile
        with tempfile.NamedTemporaryFile(delete=False, suffix=".jpg") as tmp:
            file.save(tmp.name)
            tmp_path = tmp.name
        
        try:
            # Run pipeline
            result = pipeline.predict(tmp_path, question)
            
            if result.get("success"):
                return jsonify({"answer": result["answer"], "metrics": result["metrics"]})
            else:
                return jsonify({"error": result.get("answer", "Unknown error")}), 500
        finally:
            # Cleanup
            os.unlink(tmp_path)
    
    @app.route("/metrics")
    def metrics():
        """Return aggregated pipeline metrics."""
        return jsonify(pipeline.get_metrics_summary())
    
    return app


# Example: Create and run Flask app
app = create_flask_app(pipeline)
if __name__ == "__main__":
    app.run(debug=True, port=5000)

### Example Usage

In [ ]:
# Run a prediction (uncomment with a valid image path)

# result = pipeline.predict(
#     image_source="../images/sample.jpg",
#     question="What is in this image?",
#     expected_answer=None
# )

# print(f"Answer: {result['answer']}")
# print(f"Latency: {result['metrics']['latency_ms']:.0f}ms")
# print(f"Quality Score: {result['metrics']['quality_score']}")

In [ ]:
# Batch processing example

# samples = [
#     ("../images/img1.jpg", "What objects are visible?", None),
#     ("../images/img2.jpg", "What color is the car?", "Red"),
# ]

# results = pipeline.predict_batch(samples)
# pipeline.save_results()

# # Print summary
# summary = pipeline.get_metrics_summary()
# print(f"Processed {summary.get('count', 0)} images")
# print(f"Average latency: {summary.get('avg_latency_ms', 0):.0f}ms")
# print(f"Success rate: {summary.get('success_rate', 0):.1%}")

---

## Pipeline Summary

| Stage | Component | Purpose |
|-------|-----------|----------|
| 1 | ProblemConfig | Define task type, constraints, model settings |
| 2 | DataInputLayer | Load images from files, URLs, base64 |
| 3 | PreprocessingStage | Validate, resize, normalize images |
| 4 | PromptEngineeringStage | Build task-specific prompts |
| 5 | ModelInvocationStage | Call Gemini API with error handling |
| 6 | ResponseGenerationStage | Parse raw output into structured response |
| 7 | PostProcessingStage | Clean, truncate, validate responses |
| 8 | EvaluationStage | Compute quality and performance metrics |
| 9 | StorageLoggingStage | Persist predictions and audit trails |
| 10 | UnifiedVQAPipeline | Deploy as Flask API or batch processor |

## Next Steps

1. **Customize ProblemConfig** for your specific domain
2. **Add evaluation datasets** with ground truth answers
3. **Extend prompt templates** for specialized tasks
4. **Add caching layer** for repeated images
5. **Deploy to production** using the Flask integration